In [3]:
from langgraph.graph import StateGraph,START,END

from typing import TypedDict, Annotated
from operator import add
from time import sleep
from sqlalchemy.ext.asyncio import result

# 1 定义状态
class OverAllState(TypedDict):
    logs: Annotated[list[str],add]
    # 如果出现并行节点同时更新状态往下游节点传递,必须要有reducer
    cur_id: Annotated[str,add]

# 2 定义节点
def node_1(state: OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"1k:{k},v:{v}")
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": "node_1",
    }

def node_2(state: OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"2k:{k},v:{v}")
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": "node_2",
    }

def node_3(state: OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"3k:{k},v:{v}")
    return {
        "logs": ["node_3 运行完毕"],
        "cur_id": "node_3",
    }

def node_4(state: OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"4k:{k},v:{v}")
    return {
        "logs": ["node_4 运行完毕"],
        "cur_id": "node_4",
    }

# 3 定义边
# 3.1 创建图 获取建造者
builder = StateGraph(state_schema = OverAllState)
# 3.2 添加节点
builder.add_node("node_1", node_1)
builder.add_node("node_3", node_3)
builder.add_node("node_4", node_4)
builder.add_node("node_2", node_2)

# 3.3 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", "node_4")
builder.add_edge("node_3", "node_4")
builder.add_edge("node_4", END)

graph = builder.compile()
result = graph.invoke({"logs":["start"],"cur_id":"start"})
print(result)

1k:logs,v:['start']
1k:cur_id,v:start
2k:logs,v:['start', 'node_1 运行完毕']
2k:cur_id,v:startnode_1
3k:logs,v:['start', 'node_1 运行完毕']
3k:cur_id,v:startnode_1
4k:logs,v:['start', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕']
4k:cur_id,v:startnode_1node_2node_3
{'logs': ['start', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕', 'node_4 运行完毕'], 'cur_id': 'startnode_1node_2node_3node_4'}
